# Chapter 09: Neural Network Fundamentals
**Part III — Deep Learning**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Neurons, activations, forward propagation, backpropagation, and PyTorch training loop.

## Learning Objectives
See Chapter 9 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Code example 1


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'max_features': uniform(0.3, 0.7),
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50, cv=5, scoring="f1_macro",
    n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV F1:", search.best_score_)

## Building a Multi-Layer Perceptron


In [ ]:
import torch
import torch.nn as nn

# Building a Multi-Layer Perceptron
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims)-1):
            layers.extend([
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Instantiate
model = MLP(input_dim=128, hidden_dims=[256, 256, 128], output_dim=10)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Code example 3


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            total_loss += criterion(outputs, y_batch).item()
            correct += (outputs.argmax(1) == y_batch).sum().item()
    return total_loss/len(loader), correct/len(loader.dataset)

## Activation function comparison


In [ ]:
# Activation function comparison
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-4, 4, 300)

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def tanh(x):    return np.tanh(x)
def relu(x):    return np.maximum(0, x)
def leaky_relu(x, alpha=0.1): return np.where(x > 0, x, alpha * x)
def gelu(x):    return x * sigmoid(1.702 * x)  # approx

fns = [('Sigmoid σ(x)', sigmoid, '#2E75B6'),
       ('Tanh', tanh, '#1F7D6E'),
       ('ReLU', relu, '#C0392B'),
       ('Leaky ReLU', leaky_relu, '#C7770A'),
       ('GELU', gelu, '#5B2C9E')]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, (name, fn, color) in enumerate(fns):
    y = fn(x)
    dy = np.gradient(y, x)
    axes[0,i].plot(x, y, color=color, lw=2)
    axes[0,i].axhline(0, color='k', lw=0.5); axes[0,i].axvline(0, color='k', lw=0.5)
    axes[0,i].set_title(name, fontsize=10); axes[0,i].grid(alpha=0.3)
    axes[0,i].set_ylim(-1.5, 1.5)
    axes[1,i].plot(x, dy, color=color, lw=2, linestyle='--')
    axes[1,i].axhline(0, color='k', lw=0.5); axes[1,i].axvline(0, color='k', lw=0.5)
    axes[1,i].set_title(f"d/dx {name}", fontsize=10); axes[1,i].grid(alpha=0.3)
    axes[1,i].set_ylim(-0.3, 1.2)

axes[0,0].set_ylabel('f(x)', fontsize=11)
axes[1,0].set_ylabel("f'(x) — gradient", fontsize=11)
plt.suptitle('Activation Functions and Their Gradients', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ch09_activations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sigmoid/tanh saturate (gradient → 0) causing vanishing gradients.")
print("ReLU has constant gradient for x > 0 — solves the problem but can 'die' for x < 0.")

---

## 📚 Review Questions

See Chapter 9 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 9 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*